# Local Differential Privacy (LDP): Mechanisms, Utility, and Integration

**Purpose.** Study local DP mechanisms (randomized response, Laplace, Gaussian) with controlled experiments, then show how the project configures local DP via `insurance_fl.fl.dp.LocalDPConfig` and `build_local_dp_mod`. Local DP provides per-record/client-side protection before aggregation, at a utility cost.

## Learning objectives
- Implement and analyze randomized response, Laplace, and Gaussian mechanisms.
- Quantify bias/variance/RMSE vs ε empirically and compare to theory where applicable.
- Understand local vs central DP and composition/repeated-query costs.
- Load or compute a DP utility curve artifact and interpret trade-offs.
- Use the project's `LocalDPConfig`/`build_local_dp_mod` API correctly.

## Background and math

### Local Differential Privacy
A randomized mechanism $M$ is $\varepsilon$-LDP if for all inputs $x,x'$ and outputs $y$,
$$
\Pr[M(x)=y] \le e^\varepsilon \Pr[M(x')=y].
$$
*Meaning here:* each user perturbs data locally before sharing; the server never sees raw values, but noise is typically larger than in central DP.

### Randomized response (binary)
For a binary bit $x\in\{0,1\}$, report truth with probability $p=\frac{e^\varepsilon}{1+e^\varepsilon}$ else flip. Given reports $Y_i$, an unbiased estimator of $\pi=\Pr[X=1]$ is
$$
\hat\pi = \frac{\bar Y - (1-p)}{2p-1}, \qquad p\neq \tfrac{1}{2}.
$$
*Meaning here:* we can recover population rates from noisy bits, but variance grows quickly at small ε.

### Laplace mechanism
For a function $f$ with $\ell_1$ sensitivity $\Delta_1$, release $f(x)+\mathrm{Lap}(\Delta_1/\varepsilon)$.
*Meaning here:* calibrated noise protects individual contributions to bounded queries (e.g., clipped means/sums).

### Gaussian mechanism
For $\ell_2$ sensitivity $\Delta_2$, add $\mathcal{N}(0,\sigma^2\Delta_2^2)$. A common calibration gives $(\varepsilon,\delta)$-DP for suitable $(\varepsilon,\delta,\sigma)$; classical analytic bounds are often stated for $\varepsilon<1$ (tighter accounting uses RDP/zCDP).
*Meaning here:* Gaussian noise is convenient for vector updates; reporting must include δ and the calibration assumptions.

## Assumptions and limitations
- Experiments use synthetic Bernoulli/scalar data and fixed clipping/sensitivity assumptions.
- Repeated queries compose: answering many questions per user increases total privacy loss.
- LDP protects individual records locally but may severely reduce utility for high-dimensional/complex models.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _p in [_here, _here / "notebooks", _here.parent]:
    if (_p / "_nb_common.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
        break
import _nb_common as nb
nb.style()
rng = nb.set_seed()
nb.env_report()


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Cfg:
    seed: int = nb.SEED
    n: int = 2000 if nb.MODE == "light" else 10000
    repeats: int = 40 if nb.MODE == "light" else 200
    epsilons = (0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0)
cfg = Cfg(); cfg


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from insurance_fl.fl.dp import LocalDPConfig, build_local_dp_mod

print('project API demo (disabled by default):', build_local_dp_mod(LocalDPConfig(enabled=False)))
try:
    mod = build_local_dp_mod(LocalDPConfig(enabled=True, clipping_norm=1.0, sensitivity=1.0, epsilon=2.0, delta=1e-5))
    print('enabled modifier type:', type(mod))
except Exception as e:
    print('[note] could not build LocalDpMod in this environment:', e)


## Randomized response experiment

In [ ]:
def rr_reports(x, eps, rng):
    p = np.exp(eps) / (1.0 + np.exp(eps))
    keep = rng.random(x.size) < p
    y = np.where(keep, x, 1 - x)
    est = (y.mean() - (1 - p)) / (2 * p - 1)
    return y, est, p

# true rate
pi_true = 0.3
X = (rng.random(cfg.n) < pi_true).astype(int)
rows = []
for eps in cfg.epsilons:
    ests = []
    for r in range(cfg.repeats):
        rr = np.random.default_rng(nb.SEED + r + int(eps*1000))
        _, est, p = rr_reports(X, eps, rr)
        ests.append(est)
    ests = np.array(ests)
    rows.append({
        'mechanism': 'randomized_response', 'epsilon': eps,
        'bias': float(ests.mean() - pi_true),
        'variance': float(ests.var(ddof=1)) if len(ests)>1 else 0.0,
        'rmse': float(np.sqrt(np.mean((ests-pi_true)**2))),
        'p_keep': float(np.exp(eps)/(1+np.exp(eps)))
    })
rr_df = pd.DataFrame(rows)
rr_df


In [ ]:
# Figure 1: RR RMSE vs epsilon and estimate distribution
fig, axes = plt.subplots(1,2, figsize=(11,4))
axes[0].loglog(rr_df['epsilon'], rr_df['rmse'], 'o-', label='empirical RMSE')
axes[0].set_title('Randomized response: RMSE vs ε'); axes[0].set_xlabel('ε (log)'); axes[0].set_ylabel('RMSE (log)'); axes[0].grid(True, which='both', alpha=0.25)
# box-like spread via repeated estimates at selected eps
sel = [e for e in cfg.epsilons if e in (0.25,0.5,1.0,2.0,4.0,8.0)]
data = []
for eps in sel:
    ests=[]
    for r in range(min(cfg.repeats, 80)):
        rr = np.random.default_rng(nb.SEED + 10000 + r + int(eps*1000))
        _, est, _ = rr_reports(X, eps, rr); ests.append(est)
    data.append(ests)
axes[1].boxplot(data, tick_labels=[str(e) for e in sel], showfliers=False)
axes[1].axhline(pi_true, color='k', ls='--', lw=1, label=f'true π={pi_true}')
axes[1].set_title('RR estimate distribution by ε'); axes[1].set_xlabel('ε'); axes[1].set_ylabel('Estimated rate'); axes[1].legend()
fig.tight_layout(); nb.save_fig(fig, 'local_dp_rr_rmse_box'); plt.show()


**How to read this / what it shows.** RMSE decreases as ε increases (less noise). Boxplots show estimator spread around the true rate; at low ε variance dominates.

## Laplace and Gaussian mechanisms (clipped mean example)

In [ ]:
def laplace_mech(x, eps, sensitivity, rng):
    b = sensitivity / eps
    return x + rng.laplace(0.0, b, size=np.shape(x))

def gaussian_sigma(eps, delta, sensitivity):
    # classical sufficient condition (conservative, commonly stated for eps < 1)
    if eps <= 0: return np.inf
    return np.sqrt(2*np.log(1.25/delta)) * sensitivity / eps

# clipped mean query
vals = rng.normal(loc=50, scale=10, size=cfg.n)
lo, hi = 20, 80
clipped = np.clip(vals, lo, hi)
true_mean = float(clipped.mean())
sensitivity_mean = (hi-lo)/len(clipped)  # L1/L2 for mean of bounded values (simplified)
rows = []
for eps in cfg.epsilons:
    lap_ests=[]; gauss_ests=[]
    for r in range(cfg.repeats):
        rr = np.random.default_rng(nb.SEED + 20000 + r + int(eps*1000))
        lap_ests.append(float(laplace_mech(true_mean, eps, sensitivity_mean, rr)))
        sigma = gaussian_sigma(eps, 1e-5, sensitivity_mean)
        gauss_ests.append(float(true_mean + rr.normal(0.0, sigma)))
    for name, ests in [('laplace_mean', np.array(lap_ests)), ('gaussian_mean_classical', np.array(gauss_ests))]:
        rows.append({'mechanism': name, 'epsilon': eps, 'bias': float(ests.mean()-true_mean), 'variance': float(ests.var(ddof=1)), 'rmse': float(np.sqrt(np.mean((ests-true_mean)**2)))})
mech_df = pd.DataFrame(rows)
mech_df.head()


In [ ]:
# Figure 2: mechanism comparison RMSE and noise distributions
fig, axes = plt.subplots(1,2, figsize=(11,4))
for name, g in mech_df.groupby('mechanism'):
    axes[0].loglog(g['epsilon'], g['rmse'], 'o-', label=name)
axes[0].set_title('RMSE vs ε (clipped mean mechanisms)'); axes[0].set_xlabel('ε'); axes[0].set_ylabel('RMSE'); axes[0].legend()
# noise overlay at eps=1
eps_show = 1.0
sens = sensitivity_mean
rr = np.random.default_rng(nb.SEED)
lap_noise = rr.laplace(0, sens/eps_show, size=5000)
gauss_noise = rr.normal(0, gaussian_sigma(eps_show,1e-5,sens), size=5000)
axes[1].hist(lap_noise, bins=60, alpha=0.5, density=True, label='Laplace noise')
axes[1].hist(gauss_noise, bins=60, alpha=0.5, density=True, label='Gaussian noise (classical calib)')
axes[1].set_title(f'Noise distributions at ε={eps_show}, δ=1e-5'); axes[1].set_xlabel('Noise'); axes[1].set_ylabel('Density'); axes[1].legend()
fig.tight_layout(); nb.save_fig(fig, 'local_dp_mechanism_compare'); plt.show()


**How to read this / what it shows.** Left: error decreases with larger ε for both mechanisms (under stated calibrations). Right: noise shapes differ (Laplace heavier tails than Gaussian for comparable settings). Gaussian calibration shown uses a classical bound; for ε≥1 prefer RDP-based accounting.

In [ ]:
# Figure 3: error bars and utility curves
fig, axes = plt.subplots(1,2, figsize=(11,4))
for name, g in mech_df.groupby('mechanism'):
    axes[0].errorbar(g['epsilon'], g['rmse'], yerr=np.sqrt(g['variance'].clip(lower=0)), marker='o', capsize=3, label=name)
axes[0].set_xscale('log'); axes[0].set_yscale('log'); axes[0].set_title('RMSE with sqrt(variance) bars'); axes[0].set_xlabel('ε'); axes[0].set_ylabel('RMSE'); axes[0].legend()
# utility proxy: 1/(1+RMSE)
for name, g in mech_df.groupby('mechanism'):
    util = 1.0/(1.0+g['rmse'].values)
    axes[1].plot(g['epsilon'].values, util, marker='o', label=name)
axes[1].set_xscale('log'); axes[1].set_title('Utility proxy 1/(1+RMSE) vs ε'); axes[1].set_xlabel('ε'); axes[1].set_ylabel('Utility proxy'); axes[1].set_ylim(0,1.05); axes[1].legend()
fig.tight_layout(); nb.save_fig(fig, 'local_dp_utility_curves'); plt.show()


**How to read this / what it shows.** Higher ε generally improves utility but weakens privacy. The utility proxy is illustrative; task-specific metrics should be used in real deployments.

## Committed utility artifact or computed light-mode curve

In [ ]:
artifact = None
try:
    artifact = nb.load_artifact(nb.REPORTS / 'dp_utility_curve.json')
    print('Loaded precomputed artifact: reports/dp_utility_curve.json')
except (FileNotFoundError, nb.StaleArtifactError) as e:
    print(f'[fallback] artifact ignored: {e}')
    artifact = None
computed = None
if artifact is None:
    # compute a light-mode local curve (labeled computed, not precomputed)
    epss = [1,2,4,8] if nb.MODE=='light' else [0.5,1,2,4,8]
    rows=[]
    for eps in epss:
        # proxy: RR RMSE on synthetic bit as local utility indicator
        ests=[]
        for r in range(30):
            rr = np.random.default_rng(nb.SEED + 30000 + r + eps)
            _, est, _ = rr_reports(X, eps, rr); ests.append(est)
        rows.append({'epsilon': eps, 'rmse_proxy': float(np.sqrt(np.mean((np.array(ests)-pi_true)**2))), 'origin': 'computed_in_notebook'})
    computed = pd.DataFrame(rows)
    print('Computed light-mode curve (artifact missing/unusable).')
artifact, computed


In [ ]:
# Figure 4: artifact/computed curve
fig, ax = plt.subplots(figsize=(7,4))
if artifact is not None:
    # best-effort plotting depending on schema
    if isinstance(artifact, dict) and 'configurations' in artifact:
        eps = [c.get('epsilon') for c in artifact['configurations']]
        loss = [c.get('train_loss') for c in artifact['configurations']]
        ax.plot(eps, loss, 'o-', label='precomputed artifact: train_loss')
        ax.set_ylabel('Train loss (artifact)'); ax.set_title('DP utility curve (precomputed artifact)')
    elif isinstance(artifact, list):
        df = pd.DataFrame(artifact); ax.plot(df.iloc[:,0], df.iloc[:,1], 'o-', label='precomputed artifact')
        ax.set_title('DP utility curve (precomputed artifact)')
    else:
        ax.text(0.5,0.5,'Artifact loaded (schema shown above)', ha='center'); ax.set_title('DP utility curve (precomputed artifact)')
elif computed is not None:
    ax.plot(computed['epsilon'], computed['rmse_proxy'], 'o-', label='computed proxy RMSE')
    ax.set_ylabel('RMSE proxy'); ax.set_title('Local DP utility curve (computed in notebook)')
ax.set_xlabel('ε'); ax.legend(); fig.tight_layout(); nb.save_fig(fig, 'local_dp_artifact_or_computed_curve'); plt.show()


**How to read this / what it shows.** If `reports/dp_utility_curve.json` exists it is plotted and labeled precomputed; otherwise a computed proxy curve is shown. Artifacts failing dataset-provenance checks (`dataset_version`/`config_hash`) are ignored with `[fallback]` and a light-mode result is computed instead. Artifact schema differences are handled defensively.

## Quantitative summary


In [ ]:
summary = pd.concat([rr_df.assign(family='RR'), mech_df.assign(family='mean')], ignore_index=True)
summary.pivot_table(index=['mechanism'], columns='epsilon', values='rmse')


## Sensitivity / ablation
- Smaller ε => stronger privacy, larger error.
- Larger sensitivity or wider clipping bounds require more noise.
- More repeated/local queries require composition accounting (total privacy loss increases).

## Limitations and failure cases
- Gaussian classical calibration shown is not sufficient alone for ε≥1; use RDP accountant for rigorous training accounting.
- LDP utility can collapse for high-dimensional gradients/models without amplification/advanced protocols.
- Per-user repeated participation compounds privacy loss.

## Conclusions and further experiments
- Mechanism choice and ε dominate the privacy-utility trade-off; empirical RMSE trends match expectations.
- Project integration is via `LocalDPConfig`/`build_local_dp_mod` (disabled by default).
- Further work: integrate with federated training loops, evaluate task accuracy vs ε, and compare with central DP + secure aggregation.

## References
- C. Dwork and A. Roth. *The Algorithmic Foundations of Differential Privacy*. 2014.
- Ú. Erlingsson, V. Pihur, A. Korolova. "RAPPOR: Randomized Aggregatable Privacy-Preserving Ordinal Response." *CCS*, 2014.
- J. C. Duchi, M. I. Jordan, M. J. Wainwright. "Local Privacy and Statistical Minimax Rates." *FOCS*, 2013.
